# 1. 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
# 선형 회귀 모델 대신 더 강력한 트리 기반 모델을 사용하기 위해 LightGBM을 임포트합니다.
from lightgbm import LGBMRegressor
# 공행성쌍 탐색 로직에서는 tqdm을 사용하지 않고 진행합니다.
# from tqdm import tqdm 
from sklearn.linear_model import LinearRegression # 베이스라인 코드를 위해 남겨둡니다.

# 2. 데이터 불러오기 및 전처리

In [2]:
# 파일 경로를 './data/' 폴더 구조에 맞게 수정했습니다.
TRAIN_PATH = './data/train.csv'
SUBMISSION_DIR = './submissions/' # 결과물을 저장할 폴더 경로

print("1. 데이터 불러오기 및 전처리 시작")
train = pd.read_csv(TRAIN_PATH)

# year, month, item_id 기준으로 value 합산 (seq만 다르다면 value 합산)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)["value"]
    .sum()
)

# year, month를 하나의 키(ym)로 묶기
monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# item_id × ym 피벗 (월별 총 무역량 매트릭스 생성)
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

print(f"  - 원본 데이터 행 수: {len(train)}")
print(f"  - Pivot Table Shape: {pivot.shape}")
print("  - Pivot Table Head:")
print(pivot.head())

1. 데이터 불러오기 및 전처리 시작
  - 원본 데이터 행 수: 10836
  - Pivot Table Shape: (100, 43)
  - Pivot Table Head:
ym         2022-01-01   2022-02-01   2022-03-01   2022-04-01   2022-05-01  \
item_id                                                                     
AANGBULD      14276.0      52347.0      53549.0          0.0      26997.0   
AHMDUILJ     242705.0     120847.0     197317.0     126142.0      71730.0   
ANWUJOKX          0.0          0.0          0.0      63580.0      81670.0   
APQGTRMF     383999.0     512813.0     217064.0     470398.0     539873.0   
ATLDMDBO  143097177.0  103568323.0  118403737.0  121873741.0  115024617.0   

ym        2022-06-01   2022-07-01  2022-08-01  2022-09-01  2022-10-01  ...  \
item_id                                                                ...   
AANGBULD     84489.0          0.0         0.0         0.0         0.0  ...   
AHMDUILJ    149138.0     186617.0    169995.0    140547.0     89292.0  ...   
ANWUJOKX     26424.0       8470.0         0.0     

# 3. 공행성쌍 탐색 함수

In [3]:
def safe_corr(x, y):
    """표준편차가 0인 경우(거래량이 변하지 않는 경우) NaN 방지"""
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def find_comovement_pairs(pivot, max_lag=6, min_nonzero=12, corr_threshold=0.4):
    """
    pivot 테이블에서 (A -> B) 공행성쌍을 찾아내는 함수
    
    튜닝 포인트:
    - max_lag: 최대 몇 개월까지 선후행 관계를 볼 것인가 (기본 6)
    - corr_threshold: 상관계수의 절댓값이 얼마 이상이어야 공행성으로 채택할 것인가 (기본 0.4)
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    results = []
    
    # tqdm 대신 단순 진행률 출력 (Jupyter에서 tqdm이 느릴 수 있음)
    n_items = len(items)

    for i, leader in enumerate(items):
        if i % 100 == 0:
            print(f"    - Leader Item 진행률: {i}/{n_items} ({leader})")
        
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            # lag = 1 ~ max_lag 탐색
            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue
                corr = safe_corr(x[:-lag], y[lag:])
                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            # 임계값 이상이면 공행성쌍으로 채택
            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    pairs = pd.DataFrame(results)
    return pairs

# corr_threshold를 베이스라인 0.4에서 0.45로 상향 조정해 첫 시도를 합니다.
# 이 값을 0.4, 0.5 등으로 변경하며 실험하세요.
CORR_THRESHOLD = 0.45 
pairs = find_comovement_pairs(pivot, corr_threshold=CORR_THRESHOLD) 

print(f"\n2. 공행성쌍 탐색 완료 (Threshold: {CORR_THRESHOLD})")
print(f"  - 탐색된 공행성쌍 수: {len(pairs)}")
print("  - 공행성쌍 Head:")
print(pairs.head())

    - Leader Item 진행률: 0/100 (AANGBULD)

2. 공행성쌍 탐색 완료 (Threshold: 0.45)
  - 탐색된 공행성쌍 수: 897
  - 공행성쌍 Head:
  leading_item_id following_item_id  best_lag  max_corr
0        AANGBULD          DEWLVASR         6  0.640221
1        AANGBULD          FTSVTTSR         3  0.531400
2        AANGBULD          GKQIJYDH         6  0.582501
3        AANGBULD          LLHREMKS         5  0.499734
4        AANGBULD          NAQIHUKZ         2  0.525490


# 4. 학습 데이터 구축 및 모델 학습

In [4]:
def build_training_data(pivot, pairs):
    """
    공행성쌍 + 시계열을 이용해 (X, y) 학습 데이터를 만드는 함수
    
    [베이스라인 Features]
      - b_t, b_t_1, a_t_lag, max_corr, best_lag
      
    [개선된 Features (예시)]
      - b_roll_mean_3: 후행 품목 B의 3개월 이동 평균
    """
    months = pivot.columns.to_list()
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_series_pd = pd.Series(b_series) # 이동 평균을 쉽게 계산하기 위해 Series로 변환

        # t+1이 존재하고, t-lag >= 0인 구간만 학습에 사용
        for t in range(max(lag, 1), n_months - 1):
            
            # 1. 베이스라인 Features (로그 변환 적용!)
            b_t = np.log1p(b_series[t])
            b_t_1 = np.log1p(b_series[t - 1])
            a_t_lag = np.log1p(a_series[t - lag])
            b_t_plus_1 = np.log1p(b_series[t + 1]) # <- Target에도 적용

            # 2. 피처 엔지니어링 (로그 변환 적용!)
            # 3개월 이동 평균을 계산 (이동 평균은 원본 값으로 계산)
            b_roll_mean_3_raw = b_series_pd.rolling(window=3, min_periods=1).mean().iloc[t]
            b_roll_mean_3 = np.log1p(b_roll_mean_3_raw)

            rows.append({
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                "b_roll_mean_3": b_roll_mean_3,
                "target": b_t_plus_1,  # <- log1p(Target)이 저장됨
            })

    df_train = pd.DataFrame(rows)
    return df_train

df_train_model = build_training_data(pivot, pairs)
print('\n3. 학습 데이터 구축 및 모델 학습')
print(f'  - 생성된 학습 데이터의 shape: {df_train_model.shape}')
print("  - 학습 데이터 Head:")
print(df_train_model.head())

# 모델 학습
# 새로운 피처 (b_roll_mean_3)를 추가했습니다.
feature_cols = ['b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag', 'b_roll_mean_3'] 

train_X = df_train_model[feature_cols].values
train_y = df_train_model["target"].values

# LightGBM 모델로 학습을 수행합니다.
reg = LGBMRegressor(random_state=42, n_estimators=500, learning_rate=0.05) 
# reg = LinearRegression() # <- 베이스라인으로 돌아가고 싶으면 이 주석을 해제하세요.
reg.fit(train_X, train_y)

print(f"  - 학습 모델: {type(reg).__name__} 학습 완료")


3. 학습 데이터 구축 및 모델 학습
  - 생성된 학습 데이터의 shape: (34478, 7)
  - 학습 데이터 Head:
         b_t      b_t_1    a_t_lag  max_corr  best_lag  b_roll_mean_3  \
0  11.560315  12.655149   9.566405  0.640221       6.0      12.164513   
1  12.415738  11.560315  10.865669  0.640221       6.0      12.308748   
2  11.549624  12.415738  10.888371  0.640221       6.0      11.929977   
3   0.000000  11.549624   0.000000  0.640221       6.0      11.668196   
4  12.130098   0.000000  10.203518  0.640221       6.0      11.475940   

      target  
0  12.415738  
1  11.549624  
2   0.000000  
3  12.130098  
4  12.372443  
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1282
[LightGBM] [Info] Number of data points in the train set: 34478, number of used features: 6
[LightGBM] [Info] Start training from score 12

# 5. 예측 및 제출 파일 생성

In [5]:
def predict(pivot, pairs, reg, feature_cols):
    """
    2025년 8월 총 무역량(value)을 예측하는 함수
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    # 가장 마지막 두 달 index (2025-7, 2025-6)
    t_last = n_months - 1 # 2025년 7월 (현재 시점)
    t_prev = n_months - 2 # 2025년 6월

    preds = []
    
    # 이동 평균 계산을 위한 데이터 프레임 준비
    pivot_series = pivot.apply(lambda x: pd.Series(x.values), axis=1)
    # 3개월 이동 평균을 미리 계산해 둡니다.
    pivot_roll_mean_3 = pivot_series.rolling(window=3, axis=1, min_periods=1).mean()

    print(f"4. 2025년 8월 예측 시작 ({len(pairs)}쌍)")
    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t_last - lag 가 0 이상인 경우만 예측 (데이터 기간 충족 확인)
        if t_last - lag < 0:
            continue

        # 1. 베이스라인 Features (로그 변환 적용!)
        b_t = np.log1p(b_series[t_last])
        b_t_1 = np.log1p(b_series[t_prev])
        a_t_lag = np.log1p(a_series[t_last - lag])
        
        # 2. 피처 엔지니어링 (로그 변환 적용!)
        b_roll_mean_3_raw = pivot_roll_mean_3.loc[follower].iloc[t_last] 
        b_roll_mean_3 = np.log1p(b_roll_mean_3_raw)
        
        X_test = np.array([[b_t, b_t_1, a_t_lag, corr, float(lag), b_roll_mean_3]])

        # (중요) 모델이 예측한 값은 log 스케일임
        y_pred_log = reg.predict(X_test)[0]
        
        # (중요) np.expm1을 사용해 원래 스케일로 복원
        y_pred = np.expm1(y_pred_log)

        # 후처리 (음수/정수 변환)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    df_pred = pd.DataFrame(preds)
    return df_pred

# 예측 실행
submission = predict(pivot, pairs, reg, feature_cols)
print("\n5. 제출 파일 Head:")
print(submission.head())

# 제출 파일을 submissions 폴더에 저장합니다. 버전 관리용 이름을 사용합니다.
# 예시: 'lgbm_corr_0.45_rollmean3_v1.csv'
submission.to_csv(f'{SUBMISSION_DIR}lgbm_corr_{str(CORR_THRESHOLD).replace(".", "")}_rollmean3_v1.csv', index=False)
print("\n✅ 최종 제출 파일이 submissions 폴더에 저장되었습니다.")

4. 2025년 8월 예측 시작 (897쌍)

5. 제출 파일 Head:
  leading_item_id following_item_id    value
0        AANGBULD          DEWLVASR   287888
1        AANGBULD          FTSVTTSR   195517
2        AANGBULD          GKQIJYDH  5196987
3        AANGBULD          LLHREMKS    31101
4        AANGBULD          NAQIHUKZ       25

✅ 최종 제출 파일이 submissions 폴더에 저장되었습니다.


/var/folders/n0/b4ggg87x0ps67bsmkvz5zjp80000gn/T/ipykernel_3959/2221691346.py:17: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  pivot_roll_mean_3 = pivot_series.rolling(window=3, axis=1, min_periods=1).mean()
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid fe